<a href="https://colab.research.google.com/github/Kylalala13/PEMROGRAMAN/blob/tugas/Jobsheet_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
%%writefile konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Belanja", "Kesehatan", "Pendidikan", "Tagihan", "Hiburan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

Overwriting konfigurasi.py


In [18]:
%%writefile setup_db_pengeluaran.py
import sqlite3
import os
from konfigurasi import DB_PATH

def setup_database():
    print(f"Memeriksa/membuat database di: {DB_PATH}")
    conn = None
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK (jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        print(" Membuat tabel 'transaksi' (jika belum ada)...")
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f" -> Error SQLite saat setup: {e}")
        return False
    finally:
        if conn:
            conn.close()
            print(" -> Koneksi DB setup ditutup.")

if __name__ == "__main__":
    print("--- Memulai Setup Database Pengeluaran ---")
    if setup_database():
        print(f"\nSetup database '{os.path.basename(DB_PATH)}' selesai.")
    else:
        print(f"\nSetup database GAGAL.")
    print("--- Setup Database Selesai ---")

Overwriting setup_db_pengeluaran.py


In [28]:
!python setup_db_pengeluaran.py

--- Memulai Setup Database Pengeluaran ---
Memeriksa/membuat database di: /content/pengeluaran_harian.db
 Membuat tabel 'transaksi' (jika belum ada)...
 -> Tabel 'transaksi' siap.
 -> Koneksi DB setup ditutup.

Setup database 'pengeluaran_harian.db' selesai.
--- Setup Database Selesai ---


In [32]:
%%writefile database.py
import sqlite3
import streamlit as st
from konfigurasi import DB_PATH

@st.cache_resource
def get_db_connection():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

# Fungsi pendukung wajib untuk penugasan backend
def execute_query(query, params=()):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(query, params)
    conn.commit()

Overwriting database.py


In [21]:
%%writefile model.py
import datetime

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran (Data Class)."""
    def __init__(self, deskripsi: str, jumlah: float, kategori: str, tanggal, id_transaksi: int | None = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"

        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
            if jumlah_float <= 0:
                print("Peringatan: Jumlah harus positif.")
        except (ValueError, TypeError):
            self.jumlah = 0.0
            print(f"Peringatan: Jumlah '{jumlah}' tidak valid.")

        self.kategori = str(kategori) if kategori else "Lainnya"

        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
                print(f"Peringatan: Format tgl '{tanggal}' salah.")
        else:
            self.tanggal = datetime.date.today()
            print(f"Peringatan: Tipe tgl '{type(tanggal)}' tidak valid.")

    def __repr__(self) -> str:
        import locale
        try:
            locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
            jml_str = locale.format_string("%.0f", self.jumlah, grouping=True)
        except:
            jml_str = f"{self.jumlah:.0f}"
        return f"Transaksi(ID: {self.id}, Jml: {jml_str}, Tgl: {self.tanggal.strftime('%Y-%m-%d')}, Kat: '{self.kategori}', Desc: '{self.deskripsi}')"

    def to_dict(self) -> dict:
        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

Overwriting model.py


In [33]:
%%writefile manajer_anggaran.py
import pandas as pd
import database

class AnggaranHarian:
    def __init__(self):
        pass

    def tambah_transaksi(self, transaksi):
        try:
            conn = database.get_db_connection()
            cursor = conn.cursor()
            cursor.execute(
                "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)",
                (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal)
            )
            conn.commit()
            return True
        except Exception as e:
            print(f"Error tambah: {e}")
            return False

    def get_dataframe_transaksi(self):
        try:
            conn = database.get_db_connection()
            df = pd.read_sql_query("SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi", conn)
            return df
        except Exception as e:
            print(f"Error read: {e}")
            return None

    # === KODE PENUGASAN: METODE HAPUS TRANSAKSI BACKEND ===
    def hapus_transaksi(self, id_transaksi: int) -> bool:
        try:
            query = "DELETE FROM transaksi WHERE id = ?"
            # Patuh pada instruksi: Menggunakan fungsi khusus database.execute_query
            database.execute_query(query, (id_transaksi,))
            return True
        except Exception as e:
            print(f"Error SQLite saat hapus: {e}")
            return False

    def hitung_total_pengeluaran(self, tanggal=None):
        try:
            conn = database.get_db_connection()
            cursor = conn.cursor()
            if tanggal:
                cursor.execute("SELECT SUM(jumlah) FROM transaksi WHERE tanggal = ?", (str(tanggal),))
            else:
                cursor.execute("SELECT SUM(jumlah) FROM transaksi")
            res = cursor.fetchone()[0]
            return float(res) if res else 0.0
        except:
            return 0.0

    def get_pengeluaran_per_kategori(self, tanggal=None):
        try:
            conn = database.get_db_connection()
            cursor = conn.cursor()
            if tanggal:
                cursor.execute("SELECT kategori, SUM(jumlah) FROM transaksi WHERE tanggal = ? GROUP BY kategori", (str(tanggal),))
            else:
                cursor.execute("SELECT kategori, SUM(jumlah) FROM transaksi GROUP BY kategori")
            rows = cursor.fetchall()
            return {row[0]: float(row[1]) for row in rows}
        except:
            return {}

Overwriting manajer_anggaran.py


In [34]:
%%writefile main_app.py
import streamlit as st
import datetime
import pandas as pd

def format_rp(angka):
    return f"Rp {angka:,.0f}".replace(",", ".")

try:
    from model import Transaksi
    from manajer_anggaran import AnggaranHarian
    from konfigurasi import KATEGORI_PENGELUARAN
except ImportError as e:
    st.error(f"Gagal mengimpor modul: {e}. Pastikan file .py lain ada.")
    st.stop()

st.set_page_config(page_title="Catatan Pengeluaran", layout="wide", initial_sidebar_state="expanded")

@st.cache_resource
def get_anggaran_manager():
    return AnggaranHarian()

anggaran = get_anggaran_manager()

def halaman_input(anggaran: AnggaranHarian):
    st.header("💸 Tambah Pengeluaran Baru")
    with st.form("form_transaksi_baru", clear_on_submit=True):
        col1, col2 = st.columns([3, 1])
        with col1: deskripsi = st.text_input("Deskripsi*", placeholder="Contoh: Makan siang")
        with col2: kategori = st.selectbox("Kategori*:", KATEGORI_PENGELUARAN, index=0)

        col3, col4 = st.columns([1, 1])
        with col3: jumlah = st.number_input("Jumlah (Rp)*:", min_value=1, step=1000, value=None, placeholder="Contoh: 25000")
        with col4: tanggal = st.date_input("Tanggal*:", value=datetime.date.today())
        submitted = st.form_submit_button("💾 Simpan Transaksi")

        if submitted:
            if not deskripsi: st.warning("Deskripsi wajib!", icon="⚠️")
            elif jumlah is None or jumlah <= 0: st.warning("Jumlah wajib!", icon="⚠️")
            else:
                with st.spinner("Menyimpan..."):
                    tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                    if anggaran.tambah_transaksi(tx):
                        st.success("✅ Berhasil disimpan!")
                        st.cache_data.clear()
                    else:
                        st.error("Gagal simpan.", icon="❌")

def halaman_riwayat(anggaran: AnggaranHarian):
    st.header("📋 Detail Semua Transaksi (Riwayat Lengkap)")

    with st.spinner("Memuat riwayat..."):
        df_transaksi = anggaran.get_dataframe_transaksi()

    if df_transaksi is None:
        st.error("Gagal ambil riwayat.")
    elif df_transaksi.empty:
        st.info("Belum ada transaksi.")
    else:
        st.dataframe(df_transaksi, use_container_width=True, hide_index=True)

        st.write("---")
        # === KODE PENUGASAN: FRONTEND MENU HAPUS TRANSAKSI ===
        st.subheader("🗑️ Menu Hapus Transaksi")

        # Patuh pada ide penugasan: Input ID & Tombol Hapus
        id_yang_dipilih = st.number_input("ID Transaksi Hapus:", min_value=1, step=1, value=1)
        tombol_hapus = st.button("Hapus Transaksi Terpilih")

        if tombol_hapus:
            st.session_state.id_target_hapus = id_yang_dipilih
            st.session_state.status_konfirmasi = True

        # Patuh pada alur implementasi logika penugasan
        if st.session_state.get('status_konfirmasi', False):
            id_del = st.session_state.id_target_hapus

            # 1. Tampilkan konfirmasi menggunakan st.warning
            st.warning(f"Apakah Anda yakin ingin menghapus transaksi dengan ID **{id_del}**?")

            # Tombol untuk konfirmasi tindakan hapus data
            if st.button("Konfirmasi Hapus", type="primary"):
                # 2. Panggil metode anggaran.hapus_transaksi(id_yang_dipilih)
                hasil_hapus = anggaran.hapus_transaksi(id_del)

                # 3. Tampilkan pesan sukses atau gagal menggunakan st.success atau st.error
                if hasil_hapus:
                    st.success(f"Transaksi ID {id_del} berhasil dihapus!")

                    # 4. Pastikan data diperbarui setelah penghapusan (Wajib sesuai kertas tugas)
                    st.cache_data.clear()
                    st.session_state.status_konfirmasi = False
                    st.rerun()
                else:
                    st.error(f"Gagal menghapus Transaksi ID {id_del}.")
                    st.session_state.status_konfirmasi = False

def halaman_ringkasan(anggaran: AnggaranHarian):
    st.subheader("Ringkasan Pengeluaran")
    col_filter1, col_filter2 = st.columns([1, 2])
    with col_filter1:
        pilihan_periode = st.selectbox("Filter Periode:", ["Semua Waktu", "Hari Ini"], key="filter_periode")
        tanggal_filter = datetime.date.today() if pilihan_periode == "Hari Ini" else None
        label_periode = f"({datetime.date.today().strftime('%d %b')})" if pilihan_periode == "Hari Ini" else "(Semua Waktu)"

    with col_filter2:
        total_pengeluaran = anggaran.hitung_total_pengeluaran(tanggal=tanggal_filter)
        st.metric(label=f"Total Pengeluaran {label_periode}", value=format_rp(total_pengeluaran))

    st.divider()
    st.subheader(f"Pengeluaran per Kategori {label_periode}")

    dict_per_kategori = anggaran.get_pengeluaran_per_kategori(tanggal=tanggal_filter)

    if not dict_per_kategori:
        st.info("Tidak ada data untuk periode ini.")
    else:
        try:
            data_kategori = [{"Kategori": kat, "Total": jml} for kat, jml in dict_per_kategori.items()]
            df_kategori = pd.DataFrame(data_kategori).sort_values(by="Total", ascending=False).reset_index(drop=True)
            df_kategori['Total (Rp)'] = df_kategori['Total'].apply(format_rp)

            col_kat1, col_kat2 = st.columns(2)
            with col_kat1:
                st.write("Tabel:")
                st.dataframe(df_kategori[['Kategori', 'Total (Rp)']], hide_index=True, use_container_width=True)
            with col_kat2:
                st.write("Grafik:")
                st.bar_chart(df_kategori.set_index('Kategori')['Total'], use_container_width=True)
        except Exception as e:
            st.error(f"Gagal tampilkan ringkasan: {e}")

def main():
    st.sidebar.title("💰 Catatan Pengeluaran")
    menu_pilihan = st.sidebar.radio("Pilih Menu:", ["Tambah", "Riwayat", "Ringkasan"], key="menu_utama")
    st.sidebar.markdown("---")
    st.sidebar.info("Nama: Kyla Chavela\nNIM: 4.33.25.0.12")

    manajer_anggaran = get_anggaran_manager()
    if menu_pilihan == "Tambah": halaman_input(manajer_anggaran)
    elif menu_pilihan == "Riwayat": halaman_riwayat(manajer_anggaran)
    elif menu_pilihan == "Ringkasan": halaman_ringkasan(manajer_anggaran)

if __name__ == "__main__":
    main()

Overwriting main_app.py


In [24]:
!pip install streamlit pandas -q
!npm install -g localtunnel -q

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [25]:
# Jalankan cell ini untuk menyalin kode IP (dibutuhkan sebagai password localtunnel nanti)
!wget -qO- ipv4.icanhazip.com

34.148.58.78


In [35]:
!streamlit run main_app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://cute-files-itch.loca.lt
2026-06-17 15:24:13.480 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.148.58.78:8501

  Stopping...
^C


In [36]:
!mkdir -p ~/.streamlit
!echo "[server]" > ~/.streamlit/config.toml
!echo "fileWatcherType = 'none'" >> ~/.streamlit/config.toml

In [37]:
# 1. Install library ngrok
!pip install pyngrok -q

# 2. Set token Ngrok milikmu
from pyngrok import ngrok
ngrok.set_auth_token("3FGjNQ04382SEzdyERFZwJOrpyH_2b6ZgtNGjQoUR2xf53wMb")

# 3. Matikan sesi lama yang macet
ngrok.kill()

# 4. Jalankan Streamlit di background & hubungkan ke Ngrok
import os
os.system("streamlit run main_app.py &")
public_url = ngrok.connect(8501)

print("\n=== SKSES! KLIK LINK DI BAWAH INI ===")
print(public_url)


=== SKSES! KLIK LINK DI BAWAH INI ===
NgrokTunnel: "https://fondue-tiptop-bush.ngrok-free.dev" -> "http://localhost:8501"
